# TP3 — 00. Auditoría, preparación y consolidación de datos

**Taller de Análisis de Datos I · Versión revisada para ejecutar desde `TP3/notebooks/`**

**Pregunta central:** ¿qué datos meteorológicos están disponibles y cuáles son suficientemente consistentes para comparar pronósticos de temperatura de **12 y 24 horas** contra observaciones?

Este notebook **no entrena modelos ni calcula resultados predictivos sobre el test**. Sus tareas son: inventariar 61 CSV; auditar estructura, cobertura y calidad; justificar la exclusión docente del **22–26/10/2026**; comprobar duplicados y desfases; y exportar dos datasets **estrictamente separados** más tablas para el futuro dashboard.

> **Alcance y precaución:** el inventario de 61 días incluye 48 días históricos para análisis (10/09–01/11, sin 22–26/10), cinco días excluidos y ocho días reservados (02–09/11). La auditoría de cobertura de noviembre **no autoriza utilizar sus observaciones/errores para seleccionar variables o modelos**. El análisis temporal causal de lags y el corte de entrenamiento corresponden al Notebook 04; esta auditoría no prueba por sí sola ausencia de leakage operativo.

**Unidad de análisis:** registro de estación, snapshot, horizonte nominal y fecha objetivo. La fecha del archivo se audita frente a la fecha objetivo de cada registro. Los CSV originales contienen 52 columnas en la muestra recibida, pero el procesado conserva un subconjunto declarado de 23 columnas para esta etapa. **Error:** observado − pronosticado, positivo cuando el pronóstico subestima la temperatura.

**Requisitos:** `pandas`, `numpy`, `matplotlib`, `pyarrow`. Instalar Parquet una sola vez desde la terminal del entorno Jupyter: `python -m pip install pyarrow` (o `py -m pip install pyarrow` en Windows si corresponde).

**Ejecución:** abrir este archivo en `TP3/notebooks/` y ejecutar de arriba hacia abajo con kernel reiniciado. `tp3_config.py` debe estar en `TP3/`, al lado de `dataset/`. Se crean archivos **nuevos** en `datos_modelado/consolidados/`, `datos_modelado/test_reservado/` y `resultados/dashboard/00_calidad_datos/`; no se sobreescriben los CSV originales, modelos ni informes de otros notebooks. Las salidas de este notebook se regeneran en sus carpetas dedicadas y no se consideran aprobadas hasta comprobarlas con los 61 CSV locales.

In [ ]:
# 00.1 — Entorno y dependencias
from pathlib import Path
from datetime import datetime, timezone
import importlib.util
import hashlib
import json
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

if importlib.util.find_spec("pyarrow") is None and importlib.util.find_spec("fastparquet") is None:
    raise RuntimeError(
        "Para exportar Parquet falta instalar pyarrow (recomendado). "
        "Ejecutá 'python -m pip install pyarrow' en el entorno de este kernel, "
        "reiniciá el kernel y volvé a ejecutar desde el comienzo."
    )

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Notebook 00: análisis y consolidación, sin entrenamiento ML.")

In [ ]:
# 00.2 — Raíz estándar TP3/, manifiesto compartido y salidas dedicadas
raices = (Path.cwd().resolve(), *Path.cwd().resolve().parents)
RAIZ_PROYECTO = next(
    (p for p in raices if (p / "dataset").is_dir() and (p / "tp3_config.py").is_file()),
    None
)
if RAIZ_PROYECTO is None:
    raise FileNotFoundError(
        "No se encontró TP3/dataset/ junto a TP3/tp3_config.py. "
        "Colocá este notebook en TP3/notebooks/ y el módulo en TP3/."
    )
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

from tp3_config import obtener_particiones, HISTORICO, TEST, EXCLUIDOS

particiones = obtener_particiones(RAIZ_PROYECTO, exigir_61=True)
archivos = particiones["inventario"]
archivos_historicos = particiones["historico"]
archivos_excluidos = particiones["excluidos"]
archivos_test = particiones["test"]
DATASET_DIR = particiones["dataset_dir"]

assert (len(archivos), len(archivos_historicos), len(archivos_excluidos), len(archivos_test)) == (61, 48, 5, 8)
assert set(archivos_historicos).isdisjoint(archivos_test)
assert set(archivos_historicos).isdisjoint(archivos_excluidos)

RESULTADOS_DIR = RAIZ_PROYECTO / "resultados" / "dashboard" / "00_calidad_datos"
HISTORICO_DIR = RAIZ_PROYECTO / "datos_modelado" / "consolidados"
TEST_DIR = RAIZ_PROYECTO / "datos_modelado" / "test_reservado"
RUN_ID = datetime.now(timezone.utc).strftime("tp3_00_%Y%m%dT%H%M%SZ")

print("Raíz:", RAIZ_PROYECTO)
print("Archivos totales:", len(archivos), "| históricos:", len(archivos_historicos),
      "| excluidos:", len(archivos_excluidos), "| test:", len(archivos_test))
print("Histórico:", archivos_historicos[0].name, "→", archivos_historicos[-1].name)
print("Test reservado:", archivos_test[0].name, "→", archivos_test[-1].name)
print("Identificador de ejecución:", RUN_ID)

## 1. Inventario y esquema

La fecha del nombre del archivo identifica su **partición declarada**, pero no sustituye a la marca `target_time_utc` de cada fila. La auditoría verificará ambas. El examen de encabezados permite confirmar presencia y denominación de columnas, **no certifica unidades ni calidad de todos sus valores**.

In [ ]:
# 00.3 — Esquema de los 61 archivos y muestra del primer día
COLUMNAS_CONSOLIDADAS = [
    "station_id", "state", "latitude", "longitude",
    "snapshot_time_utc", "collection_time_utc", "forecast_generated_at_utc",
    "target_time_utc", "observation_time_utc", "observation_collection_hour_utc",
    "horizon_target_hours", "match_type", "observation_time_difference_minutes",
    "temperature_forecast_c", "temperature_observed_c", "temperature_error_c",
    "relative_humidity_forecast_pct", "dewpoint_forecast_c",
    "wind_speed_forecast_kmh", "wind_direction_forecast",
    "precip_probability_forecast_pct", "short_forecast",
    "barometric_pressure_observed_pa"
]
CLAVES = ["station_id", "snapshot_time_utc", "target_time_utc", "horizon_target_hours"]
FECHAS = [
    "snapshot_time_utc", "collection_time_utc", "forecast_generated_at_utc",
    "target_time_utc", "observation_time_utc", "observation_collection_hour_utc"
]

cabecera_base = pd.read_csv(archivos_historicos[0], nrows=0).columns.tolist()
resultados_esquema = []
for archivo in archivos:
    encabezado = pd.read_csv(archivo, nrows=0).columns.tolist()
    resultados_esquema.append({
        "archivo": archivo.name,
        "columnas": len(encabezado),
        "mismas_columnas": set(encabezado) == set(cabecera_base),
        "mismo_orden": encabezado == cabecera_base,
        "faltantes_esenciales": ", ".join(sorted(set(COLUMNAS_CONSOLIDADAS) - set(encabezado))),
        "faltantes_vs_base": ", ".join(sorted(set(cabecera_base) - set(encabezado))),
        "adicionales_vs_base": ", ".join(sorted(set(encabezado) - set(cabecera_base)))
    })

df_esquema = pd.DataFrame(resultados_esquema)
print("Archivo de referencia:", archivos_historicos[0].name, "| columnas:", len(cabecera_base))
print("Esquemas idénticos:", int(df_esquema["mismas_columnas"].sum()), "de 61")
print("Diferencias detectadas:")
display(df_esquema.loc[~df_esquema["mismas_columnas"] | df_esquema["faltantes_esenciales"].ne("")])

assert df_esquema["faltantes_esenciales"].eq("").all(), "Hay columnas requeridas ausentes: revisar esquemas antes de continuar"
assert df_esquema["mismas_columnas"].all(), "Existe deriva de esquema; revisar antes de procesar"

archivo_ejemplo = archivos_historicos[0]
df_ejemplo = pd.read_csv(archivo_ejemplo, nrows=5, low_memory=False)
print("\nCinco filas de ejemplo (no representativas del histórico completo):")
display(df_ejemplo[["station_id", "horizon_target_hours", "match_type",
                   "temperature_forecast_c", "temperature_observed_c", "target_time_utc"]])

## 2. Auditoría integral por archivo, horizonte y estación

Se realiza **una sola pasada por los 61 CSV**, leyendo únicamente las columnas relevantes por bloques de 100.000 filas. Esto evita volver a cargar repetidamente el archivo de ejemplo y permite obtener indicadores reales por día.

Se distinguen cuatro estados de temperatura (`ambas disponibles`, `solo pronóstico`, `solo observación`, `ambas faltantes`) y tres denominadores distintos:

- `registros`: registros de horizontes 12/24, sin exigir observación.
- `ambas_temperaturas`: registros con dos temperaturas finitas, sin importar `match_type`.
- `interval_validos`: registros de tipo `interval` con temperaturas finitas; este es el filtro utilizado por el modelo.

Las marcas de tiempo, los duplicados y las variables meteorológicas se auditan en **histórico y excluidos** como evidencia de calidad. Los CSV reservados de noviembre se procesan para cobertura y separación, pero **no se calculan métricas de rendimiento ni se utilizan sus errores para elegir reglas**. Una cobertura baja no equivale por sí sola a error del modelo.

In [ ]:
# 00.4 — Funciones de auditoría (sin decisiones basadas en el test)
RANGOS = {
    "relative_humidity_forecast_pct": (0, 100),
    "precip_probability_forecast_pct": (0, 100),
    "wind_speed_forecast_kmh": (0, None),
    "latitude": (-90, 90),
    "longitude": (-180, 180),
    "barometric_pressure_observed_pa": (0, None)
}
VARIABLES_AUDITAR = [
    "temperature_forecast_c", "dewpoint_forecast_c", "relative_humidity_forecast_pct",
    "wind_speed_forecast_kmh", "precip_probability_forecast_pct", "latitude", "longitude",
    "barometric_pressure_observed_pa", "short_forecast", "wind_direction_forecast"
]

def fecha_de_archivo(archivo):
    return pd.to_datetime(archivo.stem.replace("forecast_observation_", ""), format="%Y_%m_%d")

def convertir_fechas(df):
    for columna in FECHAS:
        df[columna] = pd.to_datetime(df[columna], utc=True, errors="coerce")
    return df

def resumir_estaciones(df, fecha, particion):
    tabla = (df.groupby("station_id", dropna=False, observed=True)
               .agg(registros=("_valido_interval", "size"),
                    validos_interval=("_valido_interval", "sum"),
                    registros_unmatched=("_unmatched", "sum"),
                    ambas_temperaturas=("_temperaturas_validas", "sum"),
                    latitud=("latitude", "first"),
                    longitud=("longitude", "first"),
                    estado=("state", "first"))
               .reset_index())
    tabla["fecha_archivo"] = fecha
    tabla["particion"] = particion
    return tabla

def resumen_variable(serie, nombre, fecha, horizonte):
    if nombre in ("short_forecast", "wind_direction_forecast"):
        return {"fecha_archivo": fecha, "horizonte_h": horizonte,
                "variable": nombre, "total": len(serie), "faltantes": int(serie.isna().sum()),
                "infinitos": 0, "fuera_rango": np.nan, "minimo": np.nan, "maximo": np.nan}
    v = pd.to_numeric(serie, errors="coerce")
    finitos = np.isfinite(v)
    bajo, alto = RANGOS.get(nombre, (None, None))
    fuera = np.zeros(len(v), dtype=bool)
    if bajo is not None:
        fuera |= (v < bajo).fillna(False).to_numpy()
    if alto is not None:
        fuera |= (v > alto).fillna(False).to_numpy()
    return {"fecha_archivo": fecha, "horizonte_h": horizonte,
            "variable": nombre, "total": len(v),
            "faltantes": int(v.isna().sum()),
            "infinitos": int(np.isinf(v).sum()),
            "fuera_rango": int(fuera.sum()),
            "minimo": float(v[ finitos ].min()) if finitos.any() else np.nan,
            "maximo": float(v[ finitos ].max()) if finitos.any() else np.nan}

print("Funciones disponibles; ninguna modifica los originales.")

In [ ]:
# 00.5 — Una pasada sobre todo el inventario (lectura puede tardar)
# El test se deposita en una lista separada y NO se analiza su error.
filas_diarias = []
resumen_estaciones_partes = []
resumen_variables_partes = []
resumen_temporal_partes = []
fragmentos_historico = []
fragmentos_test_reservado = []

dict_particion = {x["archivo"]: x["particion"] for x in particiones["filas_manifiesto"]}

for numero, archivo in enumerate(archivos, 1):
    fecha = fecha_de_archivo(archivo)
    fecha_iso = fecha.strftime("%Y-%m-%d")
    particion = dict_particion[archivo.name]
    diarios = {h: {"registros": 0, "ambas_temperaturas": 0,
                   "solo_pronostico": 0, "solo_observacion": 0, "ambas_faltantes": 0,
                   "interval_validos": 0, "interval_sin_observacion": 0,
                   "interval_tiempo_mayor_30": 0, "interval_tiempo_faltante": 0,
                   "error_guardado_inconsistente": 0, "error_guardado_faltante": 0,
                   "fecha_objetivo_otra": 0, "claves_faltantes": 0,
                   "generacion_posterior_objetivo": 0,
                   "generacion_posterior_captura": 0,
                   "match_interval": 0, "match_fallback": 0, "match_unmatched": 0,
                   "fecha_objetivo_distinta_todas": 0, "fecha_objetivo_invalida_todas": 0}
               for h in [12, 24]}
    estaciones_archivo = []
    variables_archivo = []
    temporal_archivo = []

    for bloque in pd.read_csv(
        archivo, usecols=COLUMNAS_CONSOLIDADAS,
        chunksize=100_000, low_memory=False
    ):
        bloque = bloque.loc[bloque["horizon_target_hours"].isin([12, 24])].copy()
        if bloque.empty:
            continue
        pf = pd.to_numeric(bloque["temperature_forecast_c"], errors="coerce")
        po = pd.to_numeric(bloque["temperature_observed_c"], errors="coerce")
        pv, ov = np.isfinite(pf), np.isfinite(po)
        ambos = pv & ov
        intervalo = bloque["match_type"].eq("interval")
        bloque["_temperaturas_validas"] = ambos
        bloque["_valido_interval"] = ambos & intervalo
        bloque["_unmatched"] = bloque["match_type"].eq("unmatched")
        # Auditar el target de TODAS las filas 12/24, no solo las seleccionadas.
        fecha_objetivo_todos = pd.to_datetime(bloque["target_time_utc"], utc=True, errors="coerce")
        fuera_fecha_todos = (fecha_objetivo_todos.notna() &
                             fecha_objetivo_todos.dt.strftime("%Y-%m-%d").ne(fecha_iso))
        invalido_fecha_todos = fecha_objetivo_todos.isna()

        estaciones_archivo.append(resumir_estaciones(bloque, fecha_iso, particion))

        for h in [12, 24]:
            sel_h = bloque["horizon_target_hours"].eq(h)
            bh = bloque.loc[sel_h]
            vh = bh["_valido_interval"]
            ah = bh["_temperaturas_validas"]
            dh = diarios[h]
            dh["registros"] += len(bh)
            dh["ambas_temperaturas"] += int(ah.sum())
            pf_h = pv.loc[sel_h]
            po_h = ov.loc[sel_h]
            dh["solo_pronostico"] += int((pf_h & ~po_h).sum())
            dh["solo_observacion"] += int((~pf_h & po_h).sum())
            dh["ambas_faltantes"] += int((~pf_h & ~po_h).sum())
            dh["fecha_objetivo_distinta_todas"] += int(fuera_fecha_todos.loc[sel_h].sum())
            dh["fecha_objetivo_invalida_todas"] += int(invalido_fecha_todos.loc[sel_h].sum())
            dh["interval_validos"] += int(vh.sum())
            dh["interval_sin_observacion"] += int((bh["match_type"].eq("interval") & ~ah).sum())
            for tipo in ["interval", "fallback", "unmatched"]:
                dh[f"match_{tipo}"] += int(bh["match_type"].eq(tipo).sum())

            # Las métricas de calidad basadas en error se calculan solo en desarrollo.
            if particion != "test":
                valido = bh.loc[vh].copy()
                if not valido.empty:
                    diff = pd.to_numeric(valido["observation_time_difference_minutes"], errors="coerce").abs()
                    dh["interval_tiempo_mayor_30"] += int((diff > 30 + 1e-6).sum())
                    dh["interval_tiempo_faltante"] += int(diff.isna().sum())
                    temporal_archivo.append(pd.DataFrame({
                        "particion": particion, "horizonte_h": h, "match_type": "interval", "diferencia_abs_min": diff
                    }))
                    # Error calculado vs columna almacenada (no es un target adicional).
                    error_en_csv = pd.to_numeric(valido["temperature_error_c"], errors="coerce")
                    calculado = pd.to_numeric(valido["temperature_observed_c"], errors="coerce") - pd.to_numeric(valido["temperature_forecast_c"], errors="coerce")
                    dh["error_guardado_inconsistente"] += int((error_en_csv.notna() & ((calculado - error_en_csv).abs() > 0.001)).sum())
                    dh["error_guardado_faltante"] += int(error_en_csv.isna().sum())
                    for variable in VARIABLES_AUDITAR:
                        fila_var = resumen_variable(valido[variable], variable, fecha_iso, h)
                        fila_var["particion"] = particion
                        variables_archivo.append(fila_var)

                # Comparación interval vs fallback sobre casos con ambas temperaturas
                extra = bh.loc[ah & bh["match_type"].eq("fallback")]
                if not extra.empty:
                    temporal_archivo.append(pd.DataFrame({
                        "particion": particion, "horizonte_h": h, "match_type": "fallback",
                        "diferencia_abs_min": pd.to_numeric(extra["observation_time_difference_minutes"], errors="coerce").abs()
                    }))

        # Los registros válidos se almacenan separados por partición.
        # Las filas excluidas solo contribuyen a auditoría de cobertura/estaciones.
        if particion in ("historico", "test"):
            seleccion = bloque.loc[bloque["_valido_interval"], COLUMNAS_CONSOLIDADAS].copy()
            if not seleccion.empty:
                seleccion = convertir_fechas(seleccion)
                seleccion["archivo_origen"] = archivo.name
                fechas_objetivo = seleccion["target_time_utc"].dt.strftime("%Y-%m-%d")
                # Una coincidencia de archivo y objetivo es una condición del corpus observado.
                otras = fechas_objetivo.ne(fecha_iso)
                for h in [12, 24]:
                    diarios[h]["fecha_objetivo_otra"] += int((otras & seleccion["horizon_target_hours"].eq(h)).sum())
                for h in [12, 24]:
                    sub = seleccion.loc[seleccion["horizon_target_hours"].eq(h)]
                    diarios[h]["claves_faltantes"] += int(sub[CLAVES].isna().any(axis=1).sum())
                    diarios[h]["generacion_posterior_objetivo"] += int((sub["forecast_generated_at_utc"] > sub["target_time_utc"]).sum())
                    diarios[h]["generacion_posterior_captura"] += int((sub["forecast_generated_at_utc"] > sub["collection_time_utc"]).sum())

                if particion == "historico":
                    fragmentos_historico.append(seleccion)
                else:
                    fragmentos_test_reservado.append(seleccion)

    resumen_estaciones_partes.extend(estaciones_archivo)
    resumen_variables_partes.extend(variables_archivo)
    resumen_temporal_partes.extend(temporal_archivo)
    for h in [12, 24]:
        filas_diarias.append({"fecha_archivo": fecha_iso, "archivo": archivo.name,
                             "particion": particion, "horizonte_h": h, **diarios[h]})
    if numero % 10 == 0 or numero == len(archivos):
        print(f"Procesados {numero}/{len(archivos)} CSV")

# Resumen general; todavía NO se exporta nada.
df_cobertura_diaria = pd.DataFrame(filas_diarias).sort_values(["fecha_archivo", "horizonte_h"]).reset_index(drop=True)
for numerador, nueva in [("ambas_temperaturas", "cobertura_temperaturas_pct"),
                         ("interval_validos", "cobertura_interval_pct")]:
    df_cobertura_diaria[nueva] = np.where(
        df_cobertura_diaria["registros"] > 0,
        100 * df_cobertura_diaria[numerador] / df_cobertura_diaria["registros"], np.nan
    )

df_estaciones_diaria = pd.concat(resumen_estaciones_partes, ignore_index=True)
df_estaciones_diaria = (
    df_estaciones_diaria.groupby(["fecha_archivo", "particion", "station_id"], dropna=False, as_index=False)
    .agg(registros=("registros", "sum"), validos_interval=("validos_interval", "sum"),
         ambas_temperaturas=("ambas_temperaturas", "sum"),
         registros_unmatched=("registros_unmatched", "sum"),
         latitude=("latitud", "first"), longitude=("longitud", "first"), state=("estado", "first"))
)
df_estaciones_diaria["cobertura_interval_pct"] = np.where(
    df_estaciones_diaria["registros"] > 0,
    100 * df_estaciones_diaria["validos_interval"] / df_estaciones_diaria["registros"], np.nan
)
df_variables_diaria = pd.DataFrame(resumen_variables_partes)
df_temporal = pd.concat(resumen_temporal_partes, ignore_index=True)

print("\nFilas de auditoría diaria:", len(df_cobertura_diaria))
print("Estaciones × día:", len(df_estaciones_diaria))
print("Auditorías de variables (histórico y excluidos):", len(df_variables_diaria))
print("Registros históricos seleccionados:", sum(len(x) for x in fragmentos_historico))
print("Registros reservados en test:", sum(len(x) for x in fragmentos_test_reservado))

## 3. Cobertura, exclusiones y comparación de emparejamientos

El gráfico diario permite explicar la pérdida de información de **22–26/10**, cuya exclusión responde a la recomendación docente. Se conserva el período en la **auditoría de calidad** pero nunca se incorpora al conjunto de entrenamiento. Se marca noviembre como **reservado** para recordar que sus observaciones no pueden fundamentar decisiones de modelado.

`interval` y `fallback` se comparan usando sus desfases absolutos reales; un porcentaje alto de temperaturas disponibles en `fallback` **no implica** mejor proximidad temporal.

In [ ]:
# 00.6 — Tablas de cobertura y calidad temporal
print("COBERTURA POR PARTICIÓN Y HORIZONTE:")
resumen_cobertura = (df_cobertura_diaria.groupby(["particion", "horizonte_h"])
                     [["registros", "ambas_temperaturas", "interval_validos"]]
                     .sum().reset_index())
resumen_cobertura["interval_pct"] = 100 * resumen_cobertura["interval_validos"] / resumen_cobertura["registros"]
assert (
    df_cobertura_diaria[["ambas_temperaturas", "solo_pronostico", "solo_observacion", "ambas_faltantes"]].sum(axis=1)
    == df_cobertura_diaria["registros"]
).all(), "Las cuatro categorías de temperatura no reconcilian con los registros auditados"
print("\nESTADOS DE DISPONIBILIDAD (por partición y horizonte):")
display(df_cobertura_diaria.groupby(["particion", "horizonte_h"])[
    ["ambas_temperaturas", "solo_pronostico", "solo_observacion", "ambas_faltantes"]
].sum().reset_index())
display(resumen_cobertura.round(2))

print("\nCONTROL DE FECHAS OBJETIVO SOBRE TODAS LAS FILAS 12/24:")
display(df_cobertura_diaria[["particion", "horizonte_h", "fecha_objetivo_distinta_todas",
                            "fecha_objetivo_invalida_todas"]]
        .groupby(["particion", "horizonte_h"]).sum().reset_index())
print("\nDÍAS EXCLUIDOS (no integrados a modelado):")
display(df_cobertura_diaria.loc[df_cobertura_diaria["particion"].eq("excluido"),
    ["fecha_archivo", "horizonte_h", "registros", "ambas_temperaturas", "interval_validos", "cobertura_temperaturas_pct"]].round(2))

resumen_temporal = (df_temporal.groupby(["particion", "horizonte_h", "match_type"])
    ["diferencia_abs_min"].agg(registros="size", disponibles="count", mediana_min="median",
                                p90_min=lambda x: x.quantile(.9), maximo_min="max")
    .reset_index())
print("\nDESFASE DEL MATCH POR PARTICIÓN (histórico y excluidos, no test):")
display(resumen_temporal.round(2))

fig, ax = plt.subplots(figsize=(13, 5))
for h in [12, 24]:
    segmento = df_cobertura_diaria.loc[df_cobertura_diaria["horizonte_h"].eq(h)]
    ax.plot(pd.to_datetime(segmento["fecha_archivo"]), segmento["cobertura_temperaturas_pct"],
            marker="o", markersize=3, linewidth=1.5, label=f"{h} h")
ax.axvspan(pd.Timestamp("2026-10-22"), pd.Timestamp("2026-10-27"), alpha=.16,
           label="22–26/10: días excluidos")
ax.axvspan(pd.Timestamp("2026-11-02"), pd.Timestamp("2026-11-10"), alpha=.08,
           label="02–09/11: evaluación reservada")
ax.set(title="Disponibilidad diaria de ambas temperaturas (inventario completo)",
       ylabel="Cobertura de pronóstico y observación (%)", xlabel="Fecha")
ax.set_ylim(0,100)
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.autofmt_xdate()
fig.tight_layout()
plt.show()
plt.close(fig)

## 4. Auditoría de las variables meteorológicas y marcas temporales

Se calculan faltantes y rangos de forma **separada para histórico y días excluidos** sobre registros de tipo `interval` con ambas temperaturas finitas. La auditoría conserva faltantes en predictores, ya que la imputación y las banderas de ausencia se deciden y ajustan dentro de los pipelines del Notebook 04.

Los nombres (`_c`, `_kmh`, `_pct`, `_pa`) indican unidades declaradas, pero comprobar unidades *reales* entre fuentes requiere el diccionario/contrato del proveedor. Se detectan valores imposibles según límites generales; **no se eliminan filas automáticamente**. En particular, presión **observada** y `short_forecast` son variables descriptivas, no necesariamente predictores del modelo final. La temperatura observada y el error guardado son **etiquetas/resultados**, nunca características predictoras de la misma fila.

In [ ]:
# 00.7 — Faltantes, rangos, consistencia del error y tiempos
# Consolidación estadística ponderada: no promediar porcentajes de días sin ponderar.
def agregar_calidad_variables(tabla):
    agregado = (tabla.groupby("variable", as_index=False)
        .agg(registros=("total", "sum"), faltantes=("faltantes", "sum"),
             infinitos=("infinitos", "sum"), fuera_rango=("fuera_rango", "sum"),
             minimo=("minimo", "min"), maximo=("maximo", "max")))
    agregado["faltantes_pct"] = 100 * agregado["faltantes"] / agregado["registros"]
    return agregado

df_variables_global = agregar_calidad_variables(
    df_variables_diaria.loc[df_variables_diaria["particion"].eq("historico")]
)
df_variables_excluidas = agregar_calidad_variables(
    df_variables_diaria.loc[df_variables_diaria["particion"].eq("excluido")]
)
print("VARIABLES / FALTANTES / RANGOS (solo 48 días históricos):")
display(df_variables_global.sort_values("faltantes_pct", ascending=False).round(3))
print("\nVARIABLES DE LOS DÍAS EXCLUIDOS (auditoría descriptiva, sin modelado):")
display(df_variables_excluidas.sort_values("faltantes_pct", ascending=False).round(3))

print("\nCONTROLES HISTÓRICOS DE INTEGRIDAD:")
controles_hist = (df_cobertura_diaria.loc[df_cobertura_diaria["particion"].eq("historico")]
                 [["interval_validos", "error_guardado_inconsistente", "error_guardado_faltante",
                   "interval_tiempo_mayor_30", "interval_tiempo_faltante", "fecha_objetivo_otra",
                   "claves_faltantes", "generacion_posterior_objetivo", "generacion_posterior_captura"]]
                 .sum())
display(controles_hist.to_frame("registros"))
if controles_hist["error_guardado_inconsistente"] > 0:
    raise ValueError("Errores históricos discordantes con observado - pronosticado: revisar antes de exportar.")
if controles_hist["error_guardado_faltante"] > 0:
    print("AVISO: existen filas históricas con error almacenado ausente; el target puede recalcularse, pero debe documentarse.")
if controles_hist["interval_tiempo_mayor_30"] > 0 or controles_hist["interval_tiempo_faltante"] > 0:
    print("AVISO: casos interval con desfase >30 min o no disponible: verificar, no descartar automáticamente.")

# No confundir fecha de generación, snapshot y captura con una prueba de disponibilidad real.
print("\nLas fechas de ingestión reales no están documentadas: no se certifica ausencia operativa de leakage.")
print("Los desfases >30 min, si existen, son avisos de auditoría: investigar, no filtrar silenciosamente.")

## 5. Auditoría espacial por estación

Este maestro se calcula sobre el **histórico**, no sobre el test, para que la visualización de estaciones no dependa de resultados futuros. Una estación con más registros puede concentrarlos en pocos días: por eso se exportan también `dias_con_registros` y `dias_con_interval_validos`, además de su porcentaje de completitud.

In [ ]:
# 00.8 — Maestro de estaciones y mapa descriptivo (histórico)
res_est = df_estaciones_diaria.loc[df_estaciones_diaria["particion"].eq("historico")].copy()
res_est["dia_con_interval"] = (res_est["validos_interval"] > 0).astype(int)
df_estaciones = (res_est.groupby("station_id", as_index=False)
    .agg(state=("state", "first"), latitude=("latitude", "first"),
         longitude=("longitude", "first"), registros=("registros", "sum"),
         validos_interval=("validos_interval", "sum"),
         dias_con_registros=("fecha_archivo", "nunique"),
         dias_con_interval_validos=("dia_con_interval", "sum")))
df_estaciones["cobertura_interval_pct"] = 100 * df_estaciones["validos_interval"] / df_estaciones["registros"].replace(0, np.nan)
print("Estaciones del histórico:", df_estaciones["station_id"].nunique())
display(df_estaciones.sort_values("cobertura_interval_pct").head(12).round(2))

mapa = df_estaciones.dropna(subset=["latitude", "longitude", "cobertura_interval_pct"])
fig, ax = plt.subplots(figsize=(11, 6))
puntos = ax.scatter(mapa["longitude"], mapa["latitude"], c=mapa["cobertura_interval_pct"],
                    s=12, cmap="viridis", vmin=0, vmax=100, alpha=.85)
fig.colorbar(puntos, ax=ax, label="Cobertura de registros interval válidos (%)")
ax.set(title="Estaciones meteorológicas — cobertura del histórico", xlabel="Longitud", ylabel="Latitud")
ax.grid(alpha=.2)
fig.tight_layout()
plt.show()
plt.close(fig)

## 6. Consolidación real y duplicados entre archivos

El criterio de modelado se preserva exactamente: horizonte 12/24, `match_type == 'interval'` y temperaturas pronosticada/observada **finitas**. Se ordena por fecha objetivo UTC y se mantiene la procedencia `archivo_origen`.

**Clave para comprobar duplicados:** `station_id + snapshot_time_utc + target_time_utc + horizon_target_hours`, la usada para relacionar filas en el Notebook 04. No se deduplica de manera automática: si aparecen claves repetidas o faltantes se detiene la exportación para investigar su origen.

Se verifica que la fecha objetivo de cada fila seleccionada corresponda a la fecha declarada en el nombre de su CSV. Si no coincide, **no se borra ni se reasigna silenciosamente**: se exige reconciliar el criterio temporal con el Notebook 04 antes de cambiar la cohorte de 854.009 filas.

In [ ]:
# 00.9 — Reconciliación de filas y claves ANTES de guardar
assert fragmentos_historico, "No se obtuvieron filas históricas seleccionadas"
assert fragmentos_test_reservado, "No se obtuvieron filas de test reservadas"
df_historico = pd.concat(fragmentos_historico, ignore_index=True)
df_test_reservado = pd.concat(fragmentos_test_reservado, ignore_index=True)
del fragmentos_historico, fragmentos_test_reservado

controles = []
for nombre, tabla, esperado in [("historico", df_historico, 854009), ("test_reservado", df_test_reservado, 140822)]:
    claves_incompletas = int(tabla[CLAVES].isna().any(axis=1).sum())
    duplicados = tabla.duplicated(CLAVES, keep=False)
    extras_fecha = (tabla["target_time_utc"].dt.strftime("%Y-%m-%d")
                    != tabla["archivo_origen"].str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                         .str.replace("_", "-", regex=False))
    controles.append({"particion": nombre, "filas": len(tabla), "esperado": esperado,
                      "claves_incompletas": claves_incompletas,
                      "filas_con_clave_duplicada": int(duplicados.sum()),
                      "filas_con_objetivo_otra_fecha": int(extras_fecha.sum())})

    if claves_incompletas or duplicados.any() or extras_fecha.any():
        print(f"\nEJEMPLOS DE PROBLEMAS EN {nombre}:")
        display(tabla.loc[duplicados | extras_fecha | tabla[CLAVES].isna().any(axis=1),
                         CLAVES + ["archivo_origen"]].head(20))

df_controles = pd.DataFrame(controles)
print("CONTROLES DE CONSOLIDACIÓN:")
display(df_controles)

assert (df_controles["filas"] == df_controles["esperado"]).all(), (
    "La cohorte no coincide con el modelado previo. No sobrescribir resultados: investigar diferencias."
)
assert df_controles[["claves_incompletas", "filas_con_clave_duplicada",
                    "filas_con_objetivo_otra_fecha"]].to_numpy().sum() == 0, (
    "Hay claves o fechas problemáticas. No se exporta hasta resolver la trazabilidad."
)

assert df_historico["archivo_origen"].isin([p.name for p in archivos_historicos]).all()
assert df_test_reservado["archivo_origen"].isin([p.name for p in archivos_test]).all()
assert set(df_historico["target_time_utc"].dt.date).issubset(HISTORICO)
assert set(df_test_reservado["target_time_utc"].dt.date).issubset(TEST)

for t in (df_historico, df_test_reservado):
    assert np.isfinite(t[["temperature_forecast_c", "temperature_observed_c"]].to_numpy(dtype=float)).all()
    assert t["match_type"].eq("interval").all()
    assert t["horizon_target_hours"].isin([12,24]).all()

# No se calculan métricas ni residual corregido sobre el test en este notebook.
df_historico = df_historico.sort_values(["target_time_utc", "station_id", "horizon_target_hours"]).reset_index(drop=True)
df_test_reservado = df_test_reservado.sort_values(["target_time_utc", "station_id", "horizon_target_hours"]).reset_index(drop=True)
print("Histórico listo:", len(df_historico), "| Test reservado:", len(df_test_reservado))
print("RECUENTO POR HORIZONTE (histórico):")
conteo_historico = df_historico["horizon_target_hours"].value_counts().sort_index()
display(conteo_historico)
print("RECUENTO POR HORIZONTE (test reservado, solo conteos de selección):")
conteo_test = df_test_reservado["horizon_target_hours"].value_counts().sort_index()
display(conteo_test)
assert conteo_historico.to_dict() == {12: 427009, 24: 427000}
assert conteo_test.to_dict() == {12: 70401, 24: 70421}

## 7. Referencias temporales y efectos de las exclusiones

El horizonte nominal se define a partir del `snapshot_time_utc`; el momento de generación del proveedor y el de captura por nuestro recolector pueden diferir. La **emisión efectiva estimada** usada por el Notebook 04 será, como mínimo, el máximo de generación y captura. Aquí solo cuantificamos esa diferencia; no asumimos disponibilidad real de datos observados.

El análisis por estación calcula cuántas estaciones tuvieron **cero registros `interval` válidos**, cuántas no tuvieron ninguna fila con ambas temperaturas y cuántas registraron exclusivamente `unmatched` durante el período de caída de cobertura. Con esto podemos explicar si fue una falla concentrada o de alcance amplio sin inferir una causa técnica no documentada.

In [ ]:
# 00.10 — Referencias temporales y estaciones afectadas por octubre
referencias = df_historico[["horizon_target_hours", "snapshot_time_utc",
    "forecast_generated_at_utc", "collection_time_utc", "target_time_utc"]].copy()
referencias["lead_snapshot_h"] = (referencias["target_time_utc"] - referencias["snapshot_time_utc"]).dt.total_seconds() / 3600
referencias["lead_generacion_h"] = (referencias["target_time_utc"] - referencias["forecast_generated_at_utc"]).dt.total_seconds() / 3600
referencias["demora_captura_min"] = (referencias["collection_time_utc"] - referencias["snapshot_time_utc"]).dt.total_seconds() / 60
referencias["generacion_menos_captura_min"] = (referencias["forecast_generated_at_utc"] - referencias["collection_time_utc"]).dt.total_seconds() / 60
referencias["emision_estimada_utc"] = referencias[["collection_time_utc", "forecast_generated_at_utc"]].max(axis=1)
referencias["lead_emision_estimada_h"] = (referencias["target_time_utc"] - referencias["emision_estimada_utc"]).dt.total_seconds() / 3600
referencias["desfase_horizonte_nominal_h"] = (referencias["lead_snapshot_h"] - referencias["horizon_target_hours"]).abs()

df_temporal_referencias = (referencias.groupby("horizon_target_hours")
    [["lead_snapshot_h", "lead_generacion_h", "lead_emision_estimada_h",
      "demora_captura_min", "generacion_menos_captura_min", "desfase_horizonte_nominal_h"]]
    .agg(["count", "median", "min", "max"]))
df_temporal_referencias.columns = [f"{variable}_{estadistico}" for variable, estadistico in df_temporal_referencias.columns]
df_temporal_referencias = df_temporal_referencias.reset_index().rename(columns={"horizon_target_hours":"horizonte_h"})
print("REFERENCIAS TEMPORALES HISTÓRICAS:")
display(df_temporal_referencias.round(3))

# Cero interval válidos no equivale necesariamente a ausencia de todas las observaciones.
impacto = df_estaciones_diaria.copy()
impacto["sin_interval_valido"] = impacto["validos_interval"].eq(0)
impacto["sin_ambas_temperaturas"] = impacto["ambas_temperaturas"].eq(0)
impacto["totalmente_unmatched"] = impacto["registros_unmatched"].eq(impacto["registros"])
df_estaciones_anomalia = (impacto.groupby(["fecha_archivo", "particion"], as_index=False)
    .agg(estaciones_registradas=("station_id", "nunique"),
         estaciones_sin_interval_valido=("sin_interval_valido", "sum"),
         estaciones_sin_ambas_temperaturas=("sin_ambas_temperaturas", "sum"),
         estaciones_totalmente_unmatched=("totalmente_unmatched", "sum"),
         registros=("registros", "sum"),
         interval_validos=("validos_interval", "sum")))
df_estaciones_anomalia["estaciones_sin_interval_pct"] = (
    100 * df_estaciones_anomalia["estaciones_sin_interval_valido"] / df_estaciones_anomalia["estaciones_registradas"].replace(0, np.nan)
)
print("\nIMPACTO DE LA ANOMALÍA POR ESTACIÓN, 20–28 DE OCTUBRE:")
display(df_estaciones_anomalia.loc[df_estaciones_anomalia["fecha_archivo"].between("2026-10-20", "2026-10-28")].round(2))
print("\nNota: no se conoce la causa externa específica ni el instante real de disponibilidad de las observaciones.")

## 8. Exportaciones reutilizables para el dashboard

Los archivos del Notebook 00 se almacenan en una carpeta propia; no se sobrescriben resultados de los notebooks 04–08 ni archivos de experimentos anteriores. La tabla de estaciones usa solo histórico; la cobertura diaria contiene etiquetas explícitas `historico`, `excluido` y `test` para evitar mezclas. Los dos Parquet de detalle se guardan en **carpetas diferentes**.

El `run_id`, las fechas y las huellas SHA-256 de claves permiten detectar cuándo dos tablas provienen de ejecuciones o selecciones distintas. Estas exportaciones son **datos auditados y filtrados**, no pronósticos corregidos; las predicciones individuales del modelo final se exportarán después desde el Notebook 04.

> **Precaución:** si ya existe un archivo con el nombre de salida de esta versión, el notebook lo reemplazará **solo en las carpetas nuevas indicadas**. No modifica CSV del dataset ni el `.joblib` del modelo. Conviene conservar una copia o trabajar en una carpeta de revisión hasta aprobar los resultados.

In [ ]:
# 00.11 — Guardar Parquet históricos/test separados, tablas y metadatos
for carpeta in [RESULTADOS_DIR, HISTORICO_DIR, TEST_DIR]:
    carpeta.mkdir(parents=True, exist_ok=True)

ruta_historico = HISTORICO_DIR / "tp3_historico_interval_12_24.parquet"
ruta_test = TEST_DIR / "tp3_test_reservado_interval_12_24.parquet"

df_historico.to_parquet(ruta_historico, index=False)
df_test_reservado.to_parquet(ruta_test, index=False)

# Guardar tablas livianas para alimentar visualizaciones interactivas.
exportaciones = {
    "01_inventario_particiones.csv": pd.DataFrame(particiones["filas_manifiesto"]),
    "02_auditoria_esquema.csv": df_esquema,
    "03_auditoria_cobertura_diaria.csv": df_cobertura_diaria,
    "04_auditoria_estaciones.csv": df_estaciones,
    "05_auditoria_estaciones_diaria.csv": df_estaciones_diaria,
    "06_auditoria_variables_global.csv": df_variables_global,
    "07_auditoria_variables_diaria.csv": df_variables_diaria,
    "08_auditoria_emparejamientos.csv": resumen_temporal,
    "09_auditoria_consolidacion.csv": df_controles,
    "10_auditoria_temporal_referencias.csv": df_temporal_referencias,
    "11_impacto_estaciones_octubre.csv": df_estaciones_anomalia,
    "12_auditoria_variables_excluidas.csv": df_variables_excluidas,
}
for nombre, tabla in exportaciones.items():
    tabla.to_csv(RESULTADOS_DIR / nombre, index=False, encoding="utf-8-sig")

# Comprobar que los Parquet se vuelven a leer con mismos conteos/esquema.
for ruta, original in [(ruta_historico, df_historico), (ruta_test, df_test_reservado)]:
    importado = pd.read_parquet(ruta, columns=CLAVES)
    assert len(importado) == len(original)
    assert not importado.duplicated(CLAVES).any()
    assert all(pd.api.types.is_datetime64_any_dtype(importado[c]) for c in ["snapshot_time_utc", "target_time_utc"])

# Huella de la cohorte: sobre claves ordenadas ya validadas.
def sha_claves(df):
    h = pd.util.hash_pandas_object(df[CLAVES], index=False).to_numpy(dtype="uint64")
    return hashlib.sha256(h.tobytes()).hexdigest()

kpis = {
    "run_id": RUN_ID,
    "n_archivos_total": len(archivos),
    "dias_historicos": len(archivos_historicos),
    "dias_excluidos": len(archivos_excluidos),
    "dias_test_reservado": len(archivos_test),
    "filas_historicas_interval": len(df_historico),
    "filas_test_reservado_interval": len(df_test_reservado),
    "estaciones_historicas": int(df_estaciones["station_id"].nunique()),
    "interval_diff_mediana_min_historico": float(
        df_temporal.loc[df_temporal["particion"].eq("historico") & df_temporal["match_type"].eq("interval"), "diferencia_abs_min"].median()
    ),
    "historico_claves_sha256": sha_claves(df_historico),
    "test_claves_sha256": sha_claves(df_test_reservado),
    "ventana_historica": "2026-09-10 a 2026-11-01 (excluye 2026-10-22 a 2026-10-26)",
    "ventana_test": "2026-11-02 a 2026-11-09",
    "unidad_cobertura": "registro de horizonte 12/24 del archivo diario",
    "criterio": "match_type=interval; horizonte 12/24; temperaturas finitas",
    "advertencia_temporal": "Los lags de ML usan una demora de disponibilidad supuesta; este notebook no certifica ingestión real",
    "origen_test": "reservado y separado; no usar para ajustar modelos o seleccionar variables"
}
with open(RESULTADOS_DIR / "13_auditoria_kpis_resumen.json", "w", encoding="utf-8") as f:
    json.dump(kpis, f, indent=2, ensure_ascii=False)

print("Parquet histórico:", ruta_historico)
print("Parquet test reservado:", ruta_test)
print("Tablas para dashboard:", RESULTADOS_DIR)
print("Total archivos exportados:", len(exportaciones) + 1 + 2)
print("\nKPI AUDITADOS:")
display(pd.Series({k:v for k,v in kpis.items() if not k.endswith("sha256")}).to_frame("valor"))

## 9. Gráficos de apoyo exportados

Estos PNG no reemplazan los CSV: en el dashboard final se construirán controles interactivos a partir de las **tablas auditadas**. El gráfico de cobertura y el mapa son insumos para la primera vista: *Origen, calidad y alcance del dataset*. Se conserva el período excluido visible; nunca se dibuja una serie de precisión inventada para esos días.

In [ ]:
# 00.12 — Visualizaciones estáticas reproducibles desde tablas exportadas
fig, ax = plt.subplots(figsize=(13, 5))
for h in (12, 24):
    d = df_cobertura_diaria.loc[df_cobertura_diaria["horizonte_h"].eq(h)]
    ax.plot(pd.to_datetime(d["fecha_archivo"]), d["cobertura_interval_pct"],
            marker="o", markersize=3, linewidth=1.4, label=f"{h} horas")
ax.axvspan(pd.Timestamp("2026-10-22"), pd.Timestamp("2026-10-27"), alpha=.16,
           label="Exclusión docente: 22–26/10")
ax.axvspan(pd.Timestamp("2026-11-02"), pd.Timestamp("2026-11-10"), alpha=.08,
           label="Test reservado: 02–09/11")
ax.set(title="Cobertura de registros interval válidos — 12 y 24 horas", xlabel="Fecha",
       ylabel="Interval con ambas temperaturas / registros (%)")
ax.set_ylim(0,100)
ax.grid(alpha=.25)
ax.legend(ncol=2)
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(RESULTADOS_DIR / "14_cobertura_interval_diaria.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 6))
p = ax.scatter(mapa["longitude"], mapa["latitude"], c=mapa["cobertura_interval_pct"],
               cmap="viridis", vmin=0, vmax=100, s=12, alpha=.85)
fig.colorbar(p, ax=ax, label="Cobertura histórica interval válida (%)")
ax.set(title="Mapa de estaciones — calidad de emparejamientos históricos", xlabel="Longitud", ylabel="Latitud")
ax.grid(alpha=.2)
fig.tight_layout()
fig.savefig(RESULTADOS_DIR / "15_mapa_cobertura_estaciones.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

# La tabla numérica (no la imagen) será la fuente de gráficos interactivos.
print("Dos gráficos exportados; las tablas permiten recrearlos de forma interactiva.")

## 10. Dictamen del Notebook 00 y continuidad

**Qué se verificó al ejecutar:** inventario de fechas, consistencia estructural, cobertura completa por horizonte, tasa `interval`, calidad temporal, coherencia del error histórico, registros inválidos de los predictores, coordenadas, claves incompletas y duplicados dentro y entre archivos, concordancia fecha objetivo/fecha del CSV y conteos históricos/test. También se comprueba la lectura de los Parquet exportados.

**Qué NO se demuestra aquí:** ausencia real de leakage de lags; la demora operativa real de observaciones; independencia prospectiva estricta del test; representatividad de estaciones nuevas; validez meteorológica de las unidades solo por su nombre; mérito predictivo de Ridge o HGB. Estas limitaciones deben aparecer en el dashboard.

**Decisiones estables:** excluir 22–26/10 por pérdida de información y recomendación docente; no imputar observaciones faltantes para construir la respuesta; modelar exclusivamente `interval` con temperaturas finitas y horizontes 12/24; conservar las características con ausencias para que cada pipeline gestione su imputación sin fuga.

**Conexión con el dashboard:** usar `03_auditoria_cobertura_diaria.csv`, `04_auditoria_estaciones.csv`, `05_auditoria_estaciones_diaria.csv` y `13_auditoria_kpis_resumen.json` para explicar población, exclusiones y limitaciones de cobertura. Los resultados de precisión, confianza e interpretabilidad pertenecen a los notebooks posteriores y deben identificarse con su propio período/modelo.

**Control final:** si los 61 CSV locales no coinciden con el inventario o el conjunto de 854.009/140.822 filas (incluidos sus horizontes), el notebook se detiene **antes de exportar un consolidado engañoso**. Nunca corregir automáticamente el número esperado para hacer pasar el test.

**Cambios respecto de las versiones anteriores:** se eliminó el texto duplicado, se dejó de inferir cobertura general desde el primer día y se incorporaron controles globales de clases de faltantes, objetivos, desfases, integridad del target, duplicados, estaciones y covariables. Todos los gráficos destinados al dashboard se acompañan de CSV de origen. La prueba independiente sigue siendo retrospectiva bajo supuestos operativos, según la auditoría general.